# 실험 과정
            ┌────────────┐       ┌───────────────┐
Image ───►  │CNN Backbone│──┐    │ Keypoint CSV  │
            └────────────┘  │    └───────┬───────┘
                            ▼            ▼
                     [Image Feature]  [CSV Feature]
                            └────┬─────┘
                                 ▼
                          🔗 Concat Layer
                                 ▼
                        🔽 MLP Classification Head
                                 ▼
                        🏷️ Binary Classification (Good/Bad)
### 라이브러리 임포트

In [2]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset
from sklearn.metrics import f1_score
import numpy as np
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader
from torchvision import transforms
import torch.nn as nn
import timm  # pip install timm
import torch.optim as optim
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import os
import pandas as pd
from PIL import Image
import torch.utils
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score
import seaborn as sns
import matplotlib.pyplot as plt

c:\Users\main\miniconda3\envs\PoseDetect-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# 1. 필요한 컬럼 모두 불러오기
train_df = pd.read_csv("../dataset-modification/train_pose_parsed.csv")[[
    "filename", "class_id", "x_center", "y_center", "width", "height"
]]
valid_df = pd.read_csv("../dataset-modification/valid_pose_parsed.csv")[[
    "filename", "class_id", "x_center", "y_center", "width", "height"
]]

# 2. 중복된 filename에 대해 대표값 선택 (class_id는 min, bbox는 first)
train_df_grouped = train_df.groupby("filename").agg({
    "class_id": "min",       # 또는 mode, if needed
    "x_center": "first",
    "y_center": "first",
    "width": "first",
    "height": "first"
}).reset_index()

valid_df_grouped = valid_df.groupby("filename").agg({
    "class_id": "min",
    "x_center": "first",
    "y_center": "first",
    "width": "first",
    "height": "first"
}).reset_index()

train_df = train_df_grouped
valid_df = valid_df_grouped

train_image_dir = "../dataset-modification/train-visualized/images/"
valid_image_dir = "../dataset-modification/valid-visualized/images/"

# 3. 클래스 가중치 계산
from sklearn.utils.class_weight import compute_class_weight
import torch

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=[0, 1],
    y=train_df["class_id"]
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

# 4. 확인
print("Train 클래스 분포:\n", train_df["class_id"].value_counts())
print("Class weights:", class_weights_tensor)
print("train_df columns:", train_df.columns.tolist())


Train 클래스 분포:
 0    1586
1     995
Name: class_id, dtype: int64
Class weights: tensor([0.8137, 1.2970])
train_df columns: ['filename', 'class_id', 'x_center', 'y_center', 'width', 'height']


Keypoint 전용 Dataset: KeypointPostureDataset

In [12]:
import pandas as pd
import torch
from torch.utils.data import Dataset

class KeypointPostureDataset(Dataset):
    def __init__(self, csv_path, normalize=True):
        """
        :param csv_path: keypoint 포함된 CSV 파일 경로
        :param normalize: 정규화 여부 (Z-score 방식)
        """
        self.df = pd.read_csv(csv_path)
        self.normalize = normalize

        # keypoint 컬럼 자동 수집 (x, y만 사용)
        self.kpt_columns = [col for col in self.df.columns if "_x" in col or "_y" in col]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # keypoints: torch.FloatTensor (shape: [34])
        keypoints = row[self.kpt_columns].values.astype('float32')
        keypoints = torch.tensor(keypoints)

        if self.normalize:
            keypoints = (keypoints - keypoints.mean()) / (keypoints.std() + 1e-6)

        # label: torch.LongTensor (0 or 1)
        label = torch.tensor(row["class_id"], dtype=torch.long)

        return keypoints, label

class EarlyStopping:
    def __init__(self, patience=30, delta=0.0, checkpoint_path='checkpoint.pt'):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.checkpoint_path = checkpoint_path

    def __call__(self, val_loss, model):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f"🟡 EarlyStopping counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                print("🛑 EarlyStopping triggered! Stopping training.")
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(model)
            self.counter = 0

    def save_checkpoint(self, model):
        """Validation loss가 개선될 때만 모델 저장"""
        torch.save(model.state_dict(), self.checkpoint_path)
        print(f"✅ Model saved to {self.checkpoint_path}")


In [11]:
train_kpt_dataset = KeypointPostureDataset(
    csv_path="../dataset-modification/train_pose_parsed.csv",
    normalize=True
)

train_loader = torch.utils.data.DataLoader(train_kpt_dataset, batch_size=32, shuffle=True)

# 확인
for kpt, lbl in train_loader:
    print(kpt.shape)  # e.g. (32, 34)
    print(lbl.shape)  # e.g. (32,)
    break


torch.Size([32, 34])
torch.Size([32])


1. 이미지 특징 추출 모듈

In [ ]:
import torchvision.models as models

class ImageFeatureExtractor(nn.Module):
    def __init__(self, backbone_name='resnet50', out_dim=512):
        super().__init__()
        if backbone_name == 'resnet50':
            model = models.resnet50(weights='IMAGENET1K_V1')
            modules = list(model.children())[:-1]  # remove FC
            self.backbone = nn.Sequential(*modules)
            self.out_dim = model.fc.in_features  # usually 2048
        # 추가 backbone 지원 가능
        
        self.project = nn.Linear(self.out_dim, out_dim)  # optional compression

    def forward(self, x):  # x: (B, 3, 224, 224)
        x = self.backbone(x).squeeze()
        x = self.project(x)
        return x  # (B, out_dim)


2. Keypoint Encoder

In [ ]:
class KeypointEncoder(nn.Module):
    def __init__(self, input_dim=34, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, out_dim),
            nn.ReLU()
        )

    def forward(self, x):  # x: (B, 34)
        return self.net(x)


 3. Multimodal Classifier (Fusion + Head)

In [ ]:
class MultiModalClassifier(nn.Module):
    def __init__(self, img_feat_dim=512, kp_feat_dim=128, hidden_dim=128):
        super().__init__()
        self.fusion = nn.Sequential(
            nn.Linear(img_feat_dim + kp_feat_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, 2)  # binary
        )

    def forward(self, img_feat, kp_feat):
        x = torch.cat([img_feat, kp_feat], dim=1)
        return self.fusion(x)


In [ ]:
img_encoder = ImageFeatureExtractor(backbone_name='resnet50', out_dim=512)
kp_encoder = KeypointEncoder(input_dim=34, out_dim=128)
classifier = MultiModalClassifier(img_feat_dim=512, kp_feat_dim=128, hidden_dim=128)

# forward pass 예시
img_input = torch.randn(16, 3, 224, 224)     # 이미지 입력
kp_input = torch.randn(16, 34)               # keypoint (x, y)*17

img_feat = img_encoder(img_input)           # (16, 512)
kp_feat = kp_encoder(kp_input)              # (16, 128)
logits = classifier(img_feat, kp_feat)      # (16, 2)


In [ ]:
# ✅ Multimodal Posture Classification (Image + Keypoint)

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, average_precision_score

# ---------------------------
# 1. Data Preparation
# ---------------------------

class MultimodalPostureDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row['filename'])
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        width, height = row['width'], row['height']
        keypoints = []
        for i in range(17):
            x = row[f'kp{i}_x'] / width if width else 0
            y = row[f'kp{i}_y'] / height if height else 0
            v = row[f'kp{i}_v']
            keypoints.extend([x * v, y * v])
        keypoints = torch.tensor(keypoints, dtype=torch.float32)
        label = torch.tensor(row['class_id'], dtype=torch.long)
        return image, keypoints, label

# ---------------------------
# 2. Model Definition
# ---------------------------

def get_backbone(backbone_name):
    if backbone_name == 'efficientnet_b0':
        model = models.efficientnet_b0(weights='DEFAULT')
        feature_extractor = nn.Sequential(*list(model.features), nn.AdaptiveAvgPool2d(1))
        feature_dim = model.classifier[1].in_features
    elif backbone_name == 'mobilenet_v3_large':
        model = models.mobilenet_v3_large(weights='DEFAULT')
        feature_extractor = nn.Sequential(*list(model.features), nn.AdaptiveAvgPool2d(1))
        feature_dim = model.classifier[0].in_features
    else:  # default to resnet50
        model = models.resnet50(weights='DEFAULT')
        feature_extractor = nn.Sequential(*list(model.children())[:-1])
        feature_dim = model.fc.in_features
    return feature_extractor, feature_dim

class MultimodalModel(nn.Module):
    def __init__(self, backbone_name='resnet50', num_classes=2):
        super().__init__()
        self.feature_extractor, feat_dim = get_backbone(backbone_name)
        self.image_fc = nn.Linear(feat_dim, 256)
        self.kp_fc = nn.Sequential(
            nn.Linear(34, 128),
            nn.ReLU(),
            nn.Linear(128, 64)
        )
        self.head = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, image, keypoints):
        x_img = self.feature_extractor(image).view(image.size(0), -1)
        x_img = self.image_fc(x_img)
        x_kp = self.kp_fc(keypoints)
        x = torch.cat([x_img, x_kp], dim=1)
        return self.head(x)

# ---------------------------
# 3. Train Loop
# ---------------------------

def train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=10):
    model.to(device)
    train_losses, val_losses = [], []
    f1_scores, mAPs = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct = 0, 0
        all_preds, all_labels = [], []

        for i, (img, kp, label) in enumerate(train_loader):
            img, kp, label = img.to(device), kp.to(device), label.to(device)
            optimizer.zero_grad()
            out = model(img, kp)
            loss = criterion(out, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            preds = out.argmax(1)
            correct += (preds == label).sum().item()
            all_preds.extend(preds.detach().cpu().numpy())
            all_labels.extend(label.detach().cpu().numpy())

        acc = correct / len(train_loader.dataset)
        f1 = f1_score(all_labels, all_preds, average='macro')
        probs = nn.functional.softmax(out, dim=1)
        mAP = average_precision_score(
            nn.functional.one_hot(label, num_classes=2).cpu().numpy(),
            probs.detach().cpu().numpy(),
            average='macro'
        )

        print(f"Epoch {epoch}: Train Loss={total_loss:.4f}, Acc={acc:.4f}, F1={f1:.4f}, mAP={mAP:.4f}")
        train_losses.append(total_loss)

        # Validation
        model.eval()
        val_loss, correct = 0, 0
        val_preds, val_labels = [], []
        with torch.no_grad():
            for img, kp, label in val_loader:
                img, kp, label = img.to(device), kp.to(device), label.to(device)
                out = model(img, kp)
                loss = criterion(out, label)
                val_loss += loss.item()
                preds = out.argmax(1)
                correct += (preds == label).sum().item()
                val_preds.extend(preds.detach().cpu().numpy())
                val_labels.extend(label.detach().cpu().numpy())

        val_f1 = f1_score(val_labels, val_preds, average='macro')
        # 대신 softmax 확률 기반 mAP 계산
        probs = nn.functional.softmax(out, dim=1)
        val_mAP = average_precision_score(
            nn.functional.one_hot(label, num_classes=2).cpu().numpy(),
            probs.detach().cpu().numpy(),
            average='macro'
        )

        print(f"          Val Loss={val_loss:.4f}, Acc={correct/len(val_loader.dataset):.4f}, F1={val_f1:.4f}, mAP={val_mAP:.4f}")
        val_losses.append(val_loss)
        f1_scores.append(val_f1)
        mAPs.append(val_mAP)

    # Visualization
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1)
    plt.plot(train_losses, label='Train Loss')
    plt.plot(val_losses, label='Val Loss')
    plt.legend(); plt.title("Loss")

    plt.subplot(1, 3, 2)
    plt.plot(f1_scores, label='Val F1')
    plt.legend(); plt.title("F1 Score")

    plt.subplot(1, 3, 3)
    plt.plot(mAPs, label='Val mAP')
    plt.legend(); plt.title("mAP")

    plt.tight_layout()
    plt.show()

# ---------------------------
# 4. Run Setup
# ---------------------------

train_df = pd.read_csv("../dataset-modification/train_pose_parsed.csv")
valid_df = pd.read_csv("../dataset-modification/valid_pose_parsed.csv")

keep_cols = ['filename', 'class_id', 'width', 'height'] + [f'kp{i}_{a}' for i in range(17) for a in ['x', 'y', 'v']]
train_df = train_df[keep_cols].groupby("filename").min().reset_index()
valid_df = valid_df[keep_cols].groupby("filename").min().reset_index()

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = MultimodalPostureDataset(train_df, "../dataset-modification/train-visualized/images", train_transform)
valid_dataset = MultimodalPostureDataset(valid_df, "../dataset-modification/valid-visualized/images", val_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=32)

backbones = ["resnet50", "efficientnet_b0", "mobilenet_v3_large"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for bb in backbones:
    print(f"\n--- Training with Backbone: {bb} ---")
    model = MultimodalModel(backbone_name=bb)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    train_model(model, train_loader, val_loader, criterion, optimizer, device, epochs=10)



--- Training with Backbone: resnet50 ---
